# 03 · EDA step 2: sanity checks (women's)

Before any analysis, confirm the 2025-26 data is complete, consistent, and that the target is
defined the way the plan says.

1. **Results mix:** regulation, overtime and shootout games, and how many ties get dropped
2. **Consistency:** both team rows of a game agree, and the counts make sense
3. **Coverage:** games per team, and games per month (how the season is spread across the folds)
4. **Missing values:** in the raw counts and in the pre-game features after the warm-up

Every check is an `assert` where there's a clear right answer. The rest are printed counts
to read. Only summaries are printed, never game-level rows.

In [1]:
%run ./01_pregame_features.ipynb

All leakage checks passed.


2025-26: 696 games, 41 ties
After warm-up (both teams >= 3 GP) and dropping ties: 579 games
59 diff_ features; missing values after warm-up: 1


In [2]:
GAMES_SQL = """
SELECT g.game_id, g.game_date, g.home_team_id, g.away_team_id, g.home_goals, g.away_goals,
       g.is_neutral, g.ot_periods, g.shootout_winner_team_id, g.game_type
FROM hockey.games g
WHERE g.season = %s AND g.is_played
"""
TEAM_EXTRA_SQL = """
SELECT game_id, team_id, ot_periods, shootout_won, en_goals, sh_goals, pim, penalties, rest_days
FROM features.team_games WHERE season = %s
"""

def query(conn, sql, params):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])


with psycopg.connect(settings.database_url) as conn:
    games = query(conn, GAMES_SQL, (SEASON,))
    extra = query(conn, TEAM_EXTRA_SQL, (SEASON,))
# Same games the features use: both teams D-I that season (see 01)
games = games[games["game_id"].isin(team_games["game_id"])].reset_index(drop=True)
new_cols = ["game_id", "team_id"] + [c for c in extra.columns if c not in team_games.columns]
tg = team_games.merge(extra[new_cols], on=["game_id", "team_id"], how="left", validate="one_to_one")
print(f"{len(games)} played games, {len(tg)} team-game rows, {tg['team_id'].nunique()} teams")

696 played games, 1392 team-game rows, 41 teams


## 1. Results mix

In [3]:
margin = (games["home_goals"] - games["away_goals"]).abs()
kind = pd.Series("regulation", index=games.index, name="ended in")
kind[games["ot_periods"] > 0] = "overtime"
kind[games["shootout_winner_team_id"].notna()] = "shootout"
outcome = pd.Series("decided", index=games.index, name="result")
outcome[margin == 0] = "tie"

mix = pd.crosstab(kind, outcome, margins=True, margins_name="total")
mix

result,decided,tie,total
ended in,,,
overtime,95,12,107
regulation,560,0,560
shootout,0,29,29
total,655,41,696


In [4]:
# Shootouts are recorded as ties (the NCAA counts them as ties), so they're dropped with the ties.
assert (margin[games["shootout_winner_team_id"].notna()] == 0).all(), "shootout game with a winner on goals"
# A regulation game can't end level, and overtime games are decided by one goal unless tied
assert (margin[kind == "regulation"] > 0).all(), "tie without overtime"
assert margin[(kind == "overtime")].isin([0, 1]).all(), "overtime game decided by more than 1"

ties = int((margin == 0).sum())
print(f"Ties dropped from the target: {ties} of {len(games)} ({ties / len(games):.1%})")
print(f"  of which settled by shootout: {int((kind == 'shootout').sum())}")
print(f"Overtime winners kept as wins/losses: {int(((kind == 'overtime') & (margin > 0)).sum())}")
print(f"Game types: {games['game_type'].fillna('none').value_counts().to_dict()}")

Ties dropped from the target: 41 of 696 (5.9%)
  of which settled by shootout: 29
Overtime winners kept as wins/losses: 95
Game types: {'Non-Conference Game': 224, 'ECAC Game': 133, 'Hockey East Game': 120, 'WCHA Game': 111, 'AHA Game': 84, 'NEWHA Game': 24}


## 2. Consistency between the two team rows of each game

In [5]:
per_game = tg.groupby("game_id")
assert (per_game.size() == 2).all(), "a game without exactly two team rows"
assert (per_game["is_home"].sum() == 1).all(), "a game without exactly one home row"
assert (per_game["team_id"].nunique() == 2).all(), "a team playing itself"

# Goals, shots and special teams mirror each other
both = tg.merge(tg, left_on=["game_id", "opponent_team_id"], right_on=["game_id", "team_id"], suffixes=("", "_opp"))
for mine, theirs in [("goals_for", "goals_against_opp"), ("shots_for", "shots_against_opp"),
                     ("pp_goals", "pp_goals_against_opp"), ("pp_opportunities", "times_shorthanded_opp")]:
    bad = (both[mine] != both[theirs]).sum()
    print(f"{mine:>16} vs opponent's {theirs.removesuffix('_opp'):<18} mismatches: {bad}")

# The games table and the team rows agree on the score
home_rows = tg[tg["is_home"]].set_index("game_id")
g = games.set_index("game_id")
assert (home_rows.loc[g.index, "goals_for"].values == g["home_goals"].values).all()
assert (home_rows.loc[g.index, "goals_against"].values == g["away_goals"].values).all()

# Counts that can't be negative or exceed their parts
assert (tg[["goals_for", "goals_against", "shots_for", "shots_against", "pp_goals", "pp_opportunities"]] >= 0).all().all()
print(f"PP goals > PP chances: {(tg['pp_goals'] > tg['pp_opportunities']).sum()} rows")
assert (tg["goals_for"] <= tg["shots_for"]).all(), "more goals than shots (non-D-I data leaking in?)"
print("Goals never exceed shots on goal.")
print(f"Empty-net goals > goals: {(tg['en_goals'] > tg['goals_for']).sum()} rows")

       goals_for vs opponent's goals_against      mismatches: 0
       shots_for vs opponent's shots_against      mismatches: 0
        pp_goals vs opponent's pp_goals_against   mismatches: 0
pp_opportunities vs opponent's times_shorthanded  mismatches: 0
PP goals > PP chances: 0 rows
Goals never exceed shots on goal.
Empty-net goals > goals: 0 rows


## 3. Coverage

In [6]:
gp = tg.groupby("team_id").size()
print(f"Games per team: min {gp.min()}, median {int(gp.median())}, max {gp.max()} ({len(gp)} teams)")
print(f"Teams with fewer than 20 games: {(gp < 20).sum()}")

by_month = games.assign(month=pd.to_datetime(games["game_date"]).dt.to_period("M")).groupby("month").size()
by_month.rename("games").to_frame().T

Games per team: min 18, median 35, max 41 (41 teams)
Teams with fewer than 20 games: 2


month,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03
games,30,138,153,44,179,129,23


In [7]:
# Where the locked k = 4 folds fall in the season (women's step 1) (games after warm-up and dropping ties)
from sklearn.model_selection import TimeSeriesSplit

K = 4
spans = []
for i, (train, test) in enumerate(TimeSeriesSplit(n_splits=K).split(rows), start=1):
    d = rows["game_date"].iloc[test]
    spans.append({"fold": i, "train games": len(train), "test games": len(test),
                  "test from": d.min().date(), "test to": d.max().date()})
pd.DataFrame(spans).set_index("fold")

,train games,test games,test from,test to
fold,,,,
1,119,115,2025-11-08,2025-12-05
2,234,115,2025-12-05,2026-01-17
3,349,115,2026-01-17,2026-02-07
4,464,115,2026-02-07,2026-03-22


In [8]:
rest = tg["rest_days"].dropna()
print(f"Rest days: median {rest.median():.0f}, 1 day (back-to-back) {(rest == 1).mean():.0%}, "
      f"0 days {(rest == 0).sum()} rows, over 14 days {(rest > 14).sum()} rows")
print(f"Neutral-site games: {int(games['is_neutral'].sum())}")
print(f"Home win rate (decided games): {rows['home_win'].mean():.3f} after warm-up, "
      f"{table['home_win'].mean():.3f} all games")

Rest days: median 3, 1 day (back-to-back) 46%, 0 days 0 rows, over 14 days 40 rows
Neutral-site games: 32
Home win rate (decided games): 0.544 after warm-up, 0.542 all games


## 4. Missing values

In [9]:
raw_cols = ["goals_for", "goals_against", "shots_for", "shots_against", "pp_goals",
            "pp_opportunities", "pp_goals_against", "times_shorthanded", "en_goals", "pim"]
raw_missing = tg[raw_cols].isna().sum()
print(f"Missing raw counts: {int(raw_missing.sum())}")
assert raw_missing.sum() == 0

diff_cols = [c for c in rows.columns if c.startswith("diff_")]
miss = rows[diff_cols].isna().sum()
print(f"Pre-game features after warm-up: {len(diff_cols)} columns, "
      f"{int((miss > 0).sum())} with any missing value")
miss[miss > 0].rename("missing games").to_frame() if (miss > 0).any() else None

Missing raw counts: 0
Pre-game features after warm-up: 59 columns, 1 with any missing value


,missing games
diff_st_index_last3,1


## Findings

**The data passes. Two incomplete box scores are now excluded in `01`.**

- **Fix: complete box scores only.** `01` now uses a game only if its box score matches the
  official score and its goal list matches the score. That removes two 2025-26 games whose CHN
  box score pages are incomplete:
  - Minnesota-Duluth at Wisconsin (2025-10-11): the page omits Minnesota-Duluth's 3 goals
    (it shows 4-0; both schedule pages show 4-3). Caught by this notebook's score check.
  - Lindenwood at Minnesota State (2025-10-03): the page has no scoring summary, which
    close-game shots and the first goal are built from. Caught by step 3's goal-list check.

  696 games remain, 579 after the warm-up. Rerunning step 1 changed nothing: k = 4.
- **Results mix:** 560 regulation games, 107 overtime games (95 decided, 12 ties) and 29
  shootouts. Shootouts count as ties, so **41 games (5.9%)** drop from the target (men's:
  6.9%).
- **Consistency:** both team rows of every game mirror each other (goals, shots, PP goals,
  PP chances: zero mismatches), and there are no impossible counts.
- **Coverage:** 41 teams, 18–41 games each (median 35). Two teams played fewer than 20 D-I
  games. Games per month: about 140–180 from October to February, fewer in September,
  December and March.
- **The k = 4 folds:** 115 test games each, from early November to the NCAA tournament. Fold 4
  (Feb 7 – Mar 22) holds the conference and NCAA tournaments; report it separately.
- **Missing values:** none in the raw counts or the 49 pre-game features after the warm-up.
- **Context for later steps:**
  - **Home teams win 54.4%** of decided games after the warm-up (men's: 55.4%).
  - 32 neutral-site games. 46% of team-games are back-to-backs (the day after the team's
    previous game).
